## 2. Melakukan Tfidf & Word Embedding

In [1]:
!pip install Sastrawi

In [2]:
!pip install gensim

# --- Import Library ---

In [3]:
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

1. Dataset

In [4]:
def load_data(filepath="Kompas_articles.csv"):
    df = pd.read_csv(filepath)

    if "judul" in df.columns and "isi" in df.columns:
        df["berita"] = df["judul"].astype(str) + " " + df["isi"].astype(str)
    elif "berita" in df.columns:
        df["berita"] = df["berita"].astype(str)
    else:
        df["berita"] = df.apply(lambda row: " ".join([str(x) for x in row]), axis=1)

    return df

In [5]:
df = load_data("kompas_articles.csv")

2. Prepocessing

In [6]:
stemmer = StemmerFactory().create_stemmer()
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stopwords and len(t) > 2]
    return " ".join(tokens)

def preprocess_data(df):
    df["clean"] = df["berita"].apply(clean_text)
    return df[["berita", "clean"]]

In [7]:
df_clean = preprocess_data(df)
display(df_clean.head())

,berita,clean
0,Waspadai Kelelahan Mental akibat Kebanyakan Be...,waspada lelah mental akibat banyak berita nega...
1,Pendidikan Kewirausahaan yang Merdeka SETIAP17...,didik kewirausahaan merdeka agustus masyarakat...
2,"Dokter: Olahraga Bisa Turunkan Risiko Kanker, ...",dokter olahraga turun risiko kanker asal rutin...
3,"Peristiwa Gas Air Mata Unisba, Mendikti Janjik...",peristiwa gas air mata unisba mendikti janji d...
4,Harapan dan Catatan soal Anggaran Pendidikan T...,harap catat soal anggar didik besar panjang se...


3. TFidf

In [8]:
def compute_tfidf(df, max_features=20):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(df["clean"])

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out()
    )
    tfidf_df.index = [f"Dokumen_{i+1}" for i in range(len(df))]

    print("Shape TF-IDF:", tfidf_matrix.shape)
    return tfidf_df, tfidf_matrix

In [9]:
tfidf_df, tfidf_matrix = compute_tfidf(df, max_features=20)
display(tfidf_df.head())

Shape TF-IDF: (6, 20)


,anggar,baca,benar,besar,brian,damping,didik,indonesia,jadi,kanker,kata,kewirausahaan,kompascom,kondisi,olahraga,perintah,persen,resmi,sebut,tahun
Dokumen_1,0.00000,0.131384,0.177540,0.177540,0.000000,0.000000,0.000000,0.000000,0.177540,0.000000,0.152138,0.000000,0.262767,0.630866,0.000000,0.000000,0.000000,0.000000,0.630866,0.000000
Dokumen_2,0.00000,0.000000,0.096771,0.000000,0.000000,0.000000,0.193542,0.677396,0.387083,0.000000,0.000000,0.559117,0.000000,0.000000,0.000000,0.096771,0.096771,0.000000,0.000000,0.114621
Dokumen_3,0.00000,0.067351,0.182025,0.091013,0.000000,0.000000,0.000000,0.000000,0.000000,0.788771,0.077991,0.000000,0.067351,0.107801,0.525847,0.000000,0.182025,0.000000,0.000000,0.000000
Dokumen_4,0.00000,0.087214,0.000000,0.000000,0.680927,0.680927,0.117854,0.000000,0.000000,0.000000,0.100992,0.000000,0.087214,0.000000,0.000000,0.117854,0.000000,0.139592,0.000000,0.000000
Dokumen_5,0.74923,0.054836,0.000000,0.296401,0.000000,0.000000,0.370501,0.074100,0.000000,0.000000,0.063498,0.000000,0.054836,0.000000,0.000000,0.148200,0.148200,0.000000,0.175537,0.351074


4. Melakukan Word Embedding (CBOW)

A. Bikin fungsi training model Word2Vec

In [16]:
from gensim.models import Word2Vec

def train_word2vec(df, vector_size=100, window=5, min_count=2, sg=0):
    corpus = [row.split() for row in df['clean'].values]
    model = Word2Vec(
        sentences=corpus,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=sg
    )
    return model

A. Buat fungsi explore_word2vec()

In [14]:
def explore_word2vec(model, word="indonesia", similar_to="ekonomi"):
    print("Shape Word Embedding (jumlah_kata, dimensi):", model.wv.vectors.shape)

    # vektor kata
    if word in model.wv:
        vec = model.wv[word]
        vec_df = pd.DataFrame(vec, columns=[f"Nilai Vektor ({word})"])
    else:
        vec_df = pd.DataFrame([f"Kata '{word}' tidak ada di vocab"], columns=["Info"])

    # kata mirip
    if similar_to in model.wv:
        sim_df = pd.DataFrame(model.wv.most_similar(similar_to, topn=5),
                              columns=["Kata", "Skor Similaritas"])
    else:
        sim_df = pd.DataFrame([f"Kata '{similar_to}' tidak ada di vocab"], columns=["Info"])

    return vec_df, sim_df

In [17]:
model = train_word2vec(df, vector_size=100, window=5, min_count=1, sg=0)

In [24]:
vec_df, sim_df = explore_word2vec(model, word="indonesia", similar_to="negara")
display(vec_df.head())
display(sim_df)
print("ekonomi" in model.wv.key_to_index)
print(list(model.wv.index_to_key)[:50])  # lihat 50 kata teratas

Shape Word Embedding (jumlah_kata, dimensi): (293, 100)


,Nilai Vektor (indonesia)
0,-0.000618
1,0.000391
2,0.005143
3,0.009019
4,-0.009257


,Kata,Skor Similaritas
0,peran,0.318198
1,turun,0.314451
2,syarat,0.254208
3,saudara,0.198555
4,sidang,0.196093


False
['indonesia', 'didik', 'anggar', 'jadi', 'baca', 'besar', 'kanker', 'kompascom', 'sebut', 'tahun', 'persen', 'negara', 'brian', 'penuh', 'damping', 'kewirausahaan', 'kata', 'kondisi', 'benar', 'resmi', 'perintah', 'olahraga', 'peristiwa', 'aman', 'nasional', 'jakarta', 'depok', 'menteri', 'republik', 'negatif', 'jonathans', 'triliun', 'berita', 'hari', 'prabowo', 'asal', 'miliano', 'lindung', 'wni', 'rupa', 'beri', 'unisba', 'mata', 'dapat', 'gas', 'air', 'tetap', 'kampus', 'sakit', 'momen']
